# Distress Gesture Detection — Kaggle Training Pipeline

**Run each cell in order. Do not skip cells.**

### Required Kaggle Datasets (add before running)
Go to **Add Data** in the top-right and add:
- `shahliza27/ur-fall-detection-dataset`
- `tuyenldvn/falldataset-imvia`
- `hungkhoi/skeleton-data-of-ntu-rgbd-60-dataset`
- `simuletic/cctv-weapon-dataset`
- `simuletic/cctv-atm-robbery-detection-dataset-gun-and-knife`

### Google Drive
Make sure you have a `distress_detection/` folder in your Drive root.
All outputs (models, checkpoints, processed data) are saved there.

### Enable GPU
Settings → Accelerator → **GPU P100**

---
## Step 0 — Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/distress_detection')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')
print(f'GPU available: ', end='')
import subprocess
print(subprocess.getoutput('nvidia-smi --query-gpu=name --format=csv,noheader'))

In [ ]:
# Clone project repo
REPO_URL = 'https://github.com/shrishri12062000/distress.git'
REPO_DIR = '/kaggle/working/distress-gesture-detection'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest changes...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    print('Cloning repo...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

import sys
sys.path.insert(0, REPO_DIR)
print('Repo ready.')

In [ ]:
# Install dependencies
print('Installing dependencies...')
os.system(f'pip install -q -r {REPO_DIR}/requirements_kaggle.txt')
os.system('pip install -q mediapipe datasets huggingface-hub')
print('Dependencies installed.')

In [ ]:
# Verify Kaggle datasets are attached
KAGGLE_INPUT = Path('/kaggle/input')
expected = [
    'ur-fall-detection-dataset',
    'falldataset-imvia',
    'skeleton-data-of-ntu-rgbd-60-dataset',
    'cctv-weapon-dataset',
]
print('Checking attached datasets:')
all_ok = True
for ds in expected:
    exists = (KAGGLE_INPUT / ds).exists()
    status = '  OK' if exists else '  MISSING — add via Add Data button'
    print(f'  {ds}: {status}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nAll datasets found. Ready to proceed.')
else:
    print('\nAdd missing datasets before continuing.')

---
## Step 1 — Data Preparation
Extracts skeletons from all video datasets and merges knife datasets.
**This step is slow (30–90 min) due to MediaPipe skeleton extraction.
Results are cached to Drive so it only runs once.**

In [ ]:
os.chdir(REPO_DIR)
%run kaggle/prepare_data.py

---
## Step 2 — Validate Data
Checks all processed data before training starts.

In [ ]:
%run kaggle/validate_data.py

---
## Step 3 — Train ST-GCN (Action Classifier)
Trains the distress gesture model on all skeleton datasets.
Best checkpoint saved to Drive automatically.
**~2–4 hours on P100 GPU.**

In [ ]:
%run kaggle/train_stgcn.py

---
## Step 4 — Train YOLOv8n (Knife Detector)
Fine-tunes YOLOv8 nano on merged knife surveillance datasets.
**~30–60 min on P100 GPU.**

In [ ]:
%run kaggle/train_yolo.py

---
## Step 5 — Export Models to ONNX
Exports both trained models to ONNX format for CPU inference on your laptop.

In [ ]:
%run kaggle/export_models.py

---
## Step 6 — Training Results Summary

In [ ]:
import torch
from pathlib import Path

DRIVE_ROOT  = Path('/content/drive/MyDrive/distress_detection')
MODELS_DIR  = DRIVE_ROOT / 'models'
CKPT_DIR    = DRIVE_ROOT / 'checkpoints' / 'stgcn'

print('═' * 54)
print('  Training Complete — Results')
print('═' * 54)

# ST-GCN results
best_ckpt = CKPT_DIR / 'best.pth'
if best_ckpt.exists():
    ckpt = torch.load(str(best_ckpt), map_location='cpu')
    print(f'  ST-GCN best val accuracy : {ckpt["val_acc"]:.2f}%')
    print(f'  ST-GCN trained epochs    : {ckpt["epoch"]}')
else:
    print('  ST-GCN checkpoint        : NOT FOUND')

# ONNX model sizes
print()
for name in ['stgcn.onnx', 'yolo_knife.onnx']:
    p = MODELS_DIR / name
    if p.exists():
        size_mb = p.stat().st_size / (1024 ** 2)
        print(f'  {name:<20}: {size_mb:.1f} MB  ✓')
    else:
        print(f'  {name:<20}: MISSING')

print()
print('  Next steps:')
print('  1. Download both .onnx files from:')
print(f'     {MODELS_DIR}')
print('  2. Place them in: distress-gesture-detection/models/')
print('  3. Run: python scripts/run_camera.py')
print('═' * 54)

---
## Quick Test — Run Inference on a Sample Image (Optional)

In [ ]:
# Quick sanity check: load ONNX models and run a dummy forward pass
import numpy as np
import onnxruntime as ort

stgcn_path = str(MODELS_DIR / 'stgcn.onnx')
yolo_path  = str(MODELS_DIR / 'yolo_knife.onnx')

if Path(stgcn_path).exists():
    sess   = ort.InferenceSession(stgcn_path, providers=['CPUExecutionProvider'])
    dummy  = np.zeros((1, 3, 30, 17), dtype=np.float32)
    out    = sess.run(None, {sess.get_inputs()[0].name: dummy})
    logits = out[0][0]
    probs  = np.exp(logits) / np.exp(logits).sum()
    classes = ['normal', 'help_signal', 'collapse_falling', 'fall_down']
    print('ST-GCN dummy forward pass:')
    for c, p in zip(classes, probs):
        print(f'  {c:<20}: {p:.4f}')
    print('  ST-GCN ONNX model is working correctly.')
else:
    print('ST-GCN ONNX not found — run Step 5 first.')

if Path(yolo_path).exists():
    sess  = ort.InferenceSession(yolo_path, providers=['CPUExecutionProvider'])
    dummy = np.zeros((1, 3, 640, 640), dtype=np.float32)
    out   = sess.run(None, {sess.get_inputs()[0].name: dummy})
    print(f'\nYOLOv8 dummy forward pass output shape: {out[0].shape}')
    print('  YOLOv8 ONNX model is working correctly.')
else:
    print('YOLOv8 ONNX not found — run Step 5 first.')